In [1]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime
import time
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from Buy_Presure_Scaner import *
from Volatile_4h_Scaner import *
from Gainer_List import *
from Consolidation_24_Scaner import *

In [28]:
def get_common_symbols_every_4_hours():
    while True:
        print("Fetching Buying Presure")
        print("\n")
        # Fetch the data
        binance_df = get_binance_buy_presure(min_volume_usdt=1000000, top_n=50)
        binance_df_filtered = binance_df[(binance_df['current_buy_pressure'] > 
                                        binance_df['current_sell_pressure']) & 
                                        (binance_df['momentum'] != "BUILDING") & 
                                        (binance_df['buy_pressure_trend'] > 1) & 
                                        (binance_df['momentum'] != "BUILDING") & 
                                        (binance_df['buy_pressure_trend'] > 1) & 
                                        (binance_df['volume_trend'] > 1)] 
        print("Completed Buying Pressure Scan")
        print("\n")
        print("Fetching Volatile Coins")
        print("\n")
        calculator = BinanceVolatilityCalculator()
        top_volatile = calculator.get_top_volatile_coins_4h(top_n=50)
        print("Completed Volatile Coins Scan")
        print("\n")
        print("Fetching Top Gainers")
        print("\n")
        top_gainers = fetch_binance_gainers(limit=50)
        top_gainers = display_gainers(top_gainers)
        print("Completed Top Gainers Scan")
        print("\n")
        print("Fetching Consolidation Coins")
        print("\n")
        analyzer = ConsolidationAnalyzer()
        non_consolidating_coins = analyzer.analyze_all_coins( 
                top_count=250  # Get top 100, display top 25
            )
        con_df = analyzer.display_results(non_consolidating_coins, show_count=50)
        print("Completed Consolidation Coins Scan")

        # Convert each list of symbols to sets
        binance_symbols = set(binance_df_filtered['symbol'])
        volatile_symbols = set(top_volatile['symbol'])
        gainers_symbols = set(top_gainers['symbol'])
        con_df_symbols = set(con_df['symbol'])

        # Find the common symbols across all four lists
        common_symbols = binance_symbols & volatile_symbols & gainers_symbols & con_df_symbols

        # Print the common symbols
        print("Common coins in all four lists:")
        print(common_symbols)

        # Sleep for 4 hours (14400 seconds)
        time.sleep(14400)  # 4 hours in seconds
        
        return common_symbols

In [27]:
get_common_symbols_every_4_hours()

Fetching Buying Presure


Fetching Volatile Coins


Fetching last 4 hours of 15-minute data for volatility calculation...
Analysis time: 2025-06-11 18:32:40
Found 402 USDT pairs


KeyboardInterrupt: 